In [1]:
import copy
import os, argparse, json, shutil
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1, 2, 3" # for Pytorch DistributedDataParallel(DDP) training
import torch
from torch import optim
from torch.utils.data.distributed import DistributedSampler # for Pytorch DistbutedDataParallel(DDP) training

from sfmreg.roitr.lib.utils import setup_seed
from sfmreg.roitr.configs.utils import load_config
from easydict import EasyDict as edict
from sfmreg.roitr.dataset.dataloader import get_dataset, get_dataloader
from sfmreg.roitr.model.RIGA_v2 import create_model
from sfmreg.roitr.lib.loss import OverallLoss, Evaluator, EvaluatorRegistration
from sfmreg.roitr.lib.tester import get_trainer
from sfmreg.roitr.registration.benchmark_utils import ransac_pose_estimation_correspondences


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# set up validation
config = load_config("sfmreg/roitr/configs/val/quad.yaml")
config['root'] = "data/sfmreg/Quad/ArtsQuad_dataset/pointclouds/"
config['device'] = torch.device('cuda', 0)
##########################################################
setup_seed(42) # fix the seed

config_sfmreg = edict(config)
# config_sfmreg.pretrain = "workspace/y23w47_sfmreg_combined_lo/y23w47_sfmreg_combined_lo/model_best_PIR.pth"
config_sfmreg.pretrain = 'workspace/y24w7_sfmreg_only/model_best_PIR.pth'
experiment_conf = edict(
    name = 'omniglue',  # just for interfacing
    descriptor_dim = 256,
    lr = 16e-5,
    refiner_num_heads = 1, 
    refiner_n_layers = 4, 
    refiner_descriptor_dim = 64,
    glue_detach = True,
    glue_refine = True,
    flash = True,
    fine_loss_use_mnn = True,
    iter_size = 4,
    low_overlap = False,
    # sfmreg_mode = "sim3",
)
config_sfmreg.update(experiment_conf)

In [3]:
##################################################################
# create model
model = create_model(config_sfmreg).to(config_sfmreg.device)
state = torch.load(config_sfmreg.pretrain)
model.load_state_dict(state['state_dict'])


<All keys matched successfully>

In [4]:
##################################################################
# create model 3dm+mega
model_3dm_mega = create_model(config_sfmreg).to(config_sfmreg.device)
state = torch.load('workspace/y23w47_sfmreg_combined_lo/y23w47_sfmreg_combined_lo/model_best_PIR.pth')
model_3dm_mega.load_state_dict(state['state_dict'])

<All keys matched successfully>

In [5]:
##################################################################
# create model roitr
config_roitr = edict(config)
config_roitr.pretrain = "pretrained/model_3dmatch.pth"
######## Experiment settings ###########
experiment_conf = edict(
    lr = 4e-5,
    use_glue = False,
    glue_refine = False,
)
config_roitr.update(experiment_conf)
roitr_model = create_model(config_roitr).to(config_roitr.device)
state = torch.load(config_roitr.pretrain)
roitr_model.load_state_dict({k.replace('module.', ''): v for k, v in state['state_dict'].items()})
# roitr_model.load_state_dict(state['state_dict'])

<All keys matched successfully>

In [6]:
from sfmreg.models.pred_wrapper import PreadtorWrap, calibrate_neighbors
import sys
sys.path.append("sfmreg/OverlapPredator")
from models.architectures import KPFCNN
from configs.models import architectures

#############################################
# create Predator model
config = load_config("sfmreg/OverlapPredator/configs/val/sfmreg.yaml")
config['device'] = torch.device('cuda', 0)
config_predator = edict(config)
config_predator.pretrain = "pretrained/predator-3dmatch.pth"
architecture = "indoor"
config_predator.architecture = architectures[architecture]
model_predator = PreadtorWrap(KPFCNN(config_predator).to(config_roitr.device), config_predator)
config_predator.neighborhood_limits = [225, 31, 32, 30]

In [7]:
# get datset
train_set, val_set, benchmark_set = get_dataset(config_sfmreg)
val_loader = get_dataloader(val_set,
                            sampler=None,
                            batch_size=1,
                            num_workers=config_sfmreg.num_workers,
                            shuffle=False,
                            drop_last=False)
evaluator = EvaluatorRegistration(config_sfmreg,True)
# evaluator = Evaluator(config_sfmreg)
print('start to evaluate on validation sets...')

start to evaluate on validation sets...


In [8]:
from tqdm import tqdm
import rerun as rr
import numpy as np

rr.init("sfmreg")
rec = rr.memory_recording()
num_iter = int(len(val_loader))
c_loader_iter = val_loader.__iter__()
for c_iter in tqdm(range(num_iter)):
# for c_iter in tqdm(range(100)):
    inputs = next(c_loader_iter)#.next()
    
    # if c_iter in [482,1216]:
        
    for k, v in inputs.items():
        if type(v) == list:
            inputs[k] = [item.to(config_roitr.device) for item in v]
        elif v is None or isinstance(v, str) :
            inputs[k] = v
        else:
            inputs[k] = v.to(config_roitr.device)

    
    with torch.no_grad():
        rot, trans = inputs['rot'][0], inputs['trans'][0]
        rot_np = rot.cpu().numpy()
        # print('rot mat stas',np.linalg.det(rot_np),np.cbrt(np.linalg.det(rot_np)))
        src_pcd, tgt_pcd = inputs['src_points'].contiguous(), inputs['tgt_points'].contiguous()
        src_normals, tgt_normals = inputs['src_normals'].contiguous(), inputs['tgt_normals'].contiguous()
        src_feats, tgt_feats = inputs['src_feats'].contiguous(), inputs['tgt_feats'].contiguous()
        src_raw_pcd = inputs['raw_src_pcd'].contiguous()
        
        outputs = model.forward(src_pcd, tgt_pcd, src_feats, tgt_feats, src_normals, tgt_normals, rot, trans, src_raw_pcd)
        outputs_roitr = roitr_model.forward(src_pcd, tgt_pcd, src_feats, tgt_feats, src_normals, tgt_normals, rot, trans, src_raw_pcd)
        outputs_predator = model_predator.forward(src_pcd, tgt_pcd, src_feats, tgt_feats, src_normals, tgt_normals, rot, trans, src_raw_pcd)
        evaluator_stats = evaluator(outputs, inputs)
        evaluator_stats_roitr = evaluator(outputs_roitr, inputs)
        evaluator_stats_predator = evaluator(outputs_predator, inputs)
        # print(c_iter,"Ev. stats", evaluator_stats)

            # save stuff
        torch.save(inputs, 'results_quad6k/model_input'+str(c_iter)+'.pth')
        torch.save(outputs, 'results_quad6k/model_output'+str(c_iter)+'.pth')
        torch.save(evaluator_stats, 'results_quad6k/model_stats'+str(c_iter)+'.pth')
        torch.save(outputs_roitr, 'results_quad6k/roitr_model_output'+str(c_iter)+'.pth')
        torch.save(evaluator_stats_roitr, 'results_quad6k/roitr_model_stats'+str(c_iter)+'.pth')
        torch.save(outputs_predator, 'results_quad6k/predator_model_output'+str(c_iter)+'.pth')
        torch.save(evaluator_stats_predator, 'results_quad6k/predator_model_stats'+str(c_iter)+'.pth')
            

        # pts_A = outputs['src_points'].cpu()
        # pts_B = outputs['tgt_points'].cpu()

        # corr_A = outputs['src_corr_points'].cpu()
        # corr_B = outputs['tgt_corr_points'].cpu()

        # rr.log("pcd/cloud_0",rr.Points3D(pts_A))
        # rr.log("pcd/cloud_1",rr.Points3D(pts_B))
        # rr.log("pcd/corrs",rr.LineStrips3D(np.stack((corr_A, corr_B), axis=1),radii=0.005))
        # rr.log("text/iter", rr.TextDocument(str(c_iter)))
        # rr.log("text/num_matches", rr.TextDocument(str(evaluator_stats['num_matches'])))
        # rr.log("text/rot_error", rr.TextDocument(str(evaluator_stats['rot_error'])))
        # rr.log("text/trans_error", rr.TextDocument(str(evaluator_stats['trans_error'])))
        # rr.log("text/IR", rr.TextDocument(str(evaluator_stats['IR'])))
        
        # pts_A = outputs_roitr['src_points'].cpu()
        # pts_B = outputs_roitr['tgt_points'].cpu()

        # corr_A = outputs_roitr['src_corr_points'].cpu()
        # corr_B = outputs_roitr['tgt_corr_points'].cpu()

        # rr.log("pcd_roitr/cloud_0",rr.Points3D(pts_A))
        # rr.log("pcd_roitr/cloud_1",rr.Points3D(pts_B))
        # rr.log("pcd_roitr/corrs",rr.LineStrips3D(np.stack((corr_A, corr_B), axis=1),radii=0.005))
# rec
    

  0%|          | 0/49 [00:00<?, ?it/s]/home/edanamt/sfmreg/sfmreg/pointops.py:40: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:83.)
  idx = torch.cuda.IntTensor(m, nsample).zero_()
/home/edanamt/sfmreg/sfmreg/roitr/lib/utils.py:562: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3614.)
  src_nodes = torch.matmul(src_nodes, rot.T) + trans.T
/home/edanamt/sfmreg/sfmreg/roitr/lib/loss.py:427: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. T